# Ticket categories comment-only

# Импорты

In [15]:
!pip install catboost
!pip install pymorphy3
!pip install optuna

In [16]:
import re

import joblib
import numpy as np
import optuna
import pandas as pd
import pymorphy3
from catboost import CatBoostClassifier
from nltk.corpus import stopwords
import nltk
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Загрузка и исследование данных

In [17]:
data = pd.read_excel('nerd_analytics_v25.xlsx', sheet_name='tickets')

In [18]:
chat_history = pd.read_excel('nerd_analytics_v25.xlsx', sheet_name='chat_history')

In [19]:
chat_history = chat_history[chat_history['ticket_id'].notna()][chat_history['role'] == 'client'][['ticket_id', 'message', 'created_at']].sort_values('created_at')

/tmp/ipykernel_597/2863438230.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  chat_history = chat_history[chat_history['ticket_id'].notna()][chat_history['role'] == 'client'][['ticket_id', 'message', 'created_at']].sort_values('created_at')


In [20]:
chat_history = chat_history.drop_duplicates(subset='ticket_id', keep='first')

In [21]:
data = pd.merge(data.rename(columns={'id': 'ticket_id'}), chat_history[['ticket_id', 'message']], on='ticket_id', how='inner')

In [22]:
data = data[['ticket_id', 'message', 'final_category']].copy()
data = data.dropna(subset=['message', 'final_category']).reset_index(drop=True)

data.shape

(3000, 3)

In [23]:
data.head()

,ticket_id,message,final_category
0,ccbfdf1a-deaa-456b-b8c8-38d24a512664,Не могу найти счёт за прошлый месяц,вопрос по оплате
1,84fd58d8-ce4c-4faa-897d-40f6f78cd699,Дважды списали одну сумму,запрос возврата
2,a62f284b-f586-43b0-a31a-e974cdf265aa,Менеджер был груб при общении,жалоба на сервис
3,e8273333-012f-440b-82bf-e07f36f49c22,Виджет чата не отображается,технический сбой
4,605ea9ad-4877-4ed0-85a9-877ee438d3ca,Приложение не запускается после обновления,технический сбой


In [24]:
data['final_category'].value_counts()

,count
final_category,
технический сбой,413
ошибка в данных,410
консультация,389
запрос возврата,388
вопрос по оплате,370
проблема с доступом,357
жалоба на сервис,341
запрос документов,332


In [25]:
df1 = data.copy()

In [26]:
morph = pymorphy3.MorphAnalyzer()
russian_stopwords = stopwords.words('russian')
bad_stopwords = ['не', 'нет', 'ни', 'без']

russian_stopwords = [
    word for word in russian_stopwords
    if word not in bad_stopwords
]

CLEAN_TEXT_SOURCE = r'''
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^а-яa-zё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    lemmas = []

    for word in words:
        if word not in russian_stopwords:
            lemma = morph.parse(word)[0].normal_form
            lemmas.append(lemma)

    return " ".join(lemmas)
'''

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^а-яa-zё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    lemmas = []

    for word in words:
        if word not in russian_stopwords:
            lemma = morph.parse(word)[0].normal_form
            lemmas.append(lemma)

    return " ".join(lemmas)

df1['clean_message'] = df1['message'].apply(clean_text)

In [27]:
RANDOM_STATE = 42

## Подбираем модель и гиперпараметры для решения задачи через Optuna

In [28]:
cat_df = df1[['clean_message', 'final_category']].copy()
X_all = cat_df['clean_message']
y_all = cat_df['final_category']
groups_all = cat_df['clean_message']

N_SPLITS = 5
CV_RANDOM_STATES = [RANDOM_STATE, 101, 202, 303, 404]


def build_features(train_part, valid_part, return_preprocessors=False):
    word_vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2, max_df=0.95)
    char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=20000)

    X_train_word = word_vectorizer.fit_transform(train_part['clean_message'])
    X_valid_word = word_vectorizer.transform(valid_part['clean_message'])
    X_train_char = char_vectorizer.fit_transform(train_part['clean_message'])
    X_valid_char = char_vectorizer.transform(valid_part['clean_message'])

    X_train = hstack([X_train_word, X_train_char])
    X_valid = hstack([X_valid_word, X_valid_char])

    if return_preprocessors:
        preprocessors = {
            'word_vectorizer': word_vectorizer,
            'char_vectorizer': char_vectorizer,
        }
        return X_train, X_valid, preprocessors

    return X_train, X_valid


def build_model_and_params(trial):
    model_type = trial.suggest_categorical('model_type', ['SVC', 'LogReg', 'CatBoost'])

    if model_type == 'SVC':
        params = {
            'C': trial.suggest_float('svc_C', 0.1, 5.0, log=True),
            'kernel': trial.suggest_categorical('svc_kernel', ['linear', 'rbf']),
            'gamma': 'scale',
            'class_weight': trial.suggest_categorical('svc_class_weight', [None, 'balanced']),
        }
        return model_type, params

    if model_type == 'LogReg':
        params = {
            'C': trial.suggest_float('lr_C', 0.1, 10.0, log=True),
            'solver': trial.suggest_categorical('lr_solver', ['lbfgs', 'saga']),
            'class_weight': trial.suggest_categorical('lr_class_weight', [None, 'balanced']),
            'max_iter': 3000,
            'n_jobs': -1,
            'random_state': RANDOM_STATE,
        }
        return model_type, params

    params = {
        'iterations': trial.suggest_int('cb_iterations', 200, 900),
        'depth': trial.suggest_int('cb_depth', 4, 10),
        'learning_rate': trial.suggest_float('cb_learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_int('cb_l2_leaf_reg', 1, 12),
        'random_strength': trial.suggest_float('cb_random_strength', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('cb_bagging_temperature', 0.0, 10.0),
        'border_count': trial.suggest_int('cb_border_count', 64, 255),
        'auto_class_weights': trial.suggest_categorical('cb_auto_class_weights', ['Balanced', None]),
    }
    return model_type, params


def make_model(model_type, model_params):
    if model_type == 'SVC':
        return SVC(**model_params)

    if model_type == 'LogReg':
        return LogisticRegression(**model_params)

    return CatBoostClassifier(
        **model_params,
        loss_function='MultiClass',
        eval_metric='TotalF1',
        random_state=RANDOM_STATE,
        task_type='GPU',
        verbose=False,
    )


def get_cv_scores(model_type, model_params, data, cv_random_states):
    fold_scores = []

    for cv_seed in cv_random_states:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_seed)

        for tr_fold_idx, val_fold_idx in cv.split(
            data['clean_message'],
            data['final_category'],
            groups=data['clean_message'],
        ):
            tr_part = data.iloc[tr_fold_idx]
            val_part = data.iloc[val_fold_idx]

            X_tr, X_val = build_features(tr_part, val_part)
            y_tr = tr_part['final_category']
            y_val = val_part['final_category']

            model = make_model(model_type, model_params)
            model.fit(X_tr, y_tr, verbose=False) if model_type == 'CatBoost' else model.fit(X_tr, y_tr)
            pred = model.predict(X_val).flatten() if model_type == 'CatBoost' else model.predict(X_val)

            fold_scores.append(f1_score(y_val, pred, average='macro'))

    return fold_scores


def objective(trial):
    model_type, model_params = build_model_and_params(trial)
    fold_scores = get_cv_scores(model_type, model_params, cat_df, CV_RANDOM_STATES)
    return float(np.mean(fold_scores))


sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=40)

print('BEST MEAN CV SCORE:')
print(study.best_value)
print('BEST PARAMS:')
print(study.best_params)

[I 2026-05-26 17:15:18,787] A new study created in memory with name: no-name-6326c62a-21d9-4276-9a5b-f10baaea4c04
[I 2026-05-26 17:15:57,487] Trial 0 finished with value: 0.06722980734233763 and parameters: {'model_type': 'LogReg', 'lr_C': 1.5751320499779735, 'lr_solver': 'lbfgs', 'lr_class_weight': 'balanced'}. Best is trial 0 with value: 0.06722980734233763.
[I 2026-05-26 17:16:29,624] Trial 1 finished with value: 0.07276029126521895 and parameters: {'model_type': 'LogReg', 'lr_C': 8.706020878304859, 'lr_solver': 'lbfgs', 'lr_class_weight': 'balanced'}. Best is trial 1 with value: 0.07276029126521895.
[I 2026-05-26 17:17:01,195] Trial 2 finished with value: 0.05959126960838074 and parameters: {'model_type': 'LogReg', 'lr_C': 0.38234752246751863, 'lr_solver': 'lbfgs', 'lr_class_weight': 'balanced'}. Best is trial 1 with value: 0.07276029126521895.
[I 2026-05-26 17:17:32,190] Trial 3 finished with value: 0.04595517188240162 and parameters: {'model_type': 'LogReg', 'lr_C': 1.06774827094

BEST MEAN CV SCORE:
0.07819356653707712
BEST PARAMS:
{'model_type': 'SVC', 'svc_C': 3.8700335721489627, 'svc_kernel': 'linear', 'svc_class_weight': 'balanced'}


## Обучаем лучшую модель

In [29]:
best_params = dict(study.best_params)
best_model_type = best_params.pop('model_type')

if best_model_type == 'SVC':
    model_params = {
        'C': best_params['svc_C'],
        'kernel': best_params['svc_kernel'],
        'gamma': 'scale',
        'class_weight': best_params['svc_class_weight'],
    }
elif best_model_type == 'LogReg':
    model_params = {
        'C': best_params['lr_C'],
        'solver': best_params['lr_solver'],
        'class_weight': best_params['lr_class_weight'],
        'max_iter': 3000,
        'n_jobs': -1,
        'random_state': RANDOM_STATE,
    }
else:
    model_params = {
        'iterations': best_params['cb_iterations'],
        'depth': best_params['cb_depth'],
        'learning_rate': best_params['cb_learning_rate'],
        'l2_leaf_reg': best_params['cb_l2_leaf_reg'],
        'random_strength': best_params['cb_random_strength'],
        'bagging_temperature': best_params['cb_bagging_temperature'],
        'border_count': best_params['cb_border_count'],
        'auto_class_weights': best_params['cb_auto_class_weights'],
    }

cv_results = []
y_true_all = []
y_pred_all = []

for cv_seed in CV_RANDOM_STATES:
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_seed)

    for fold, (tr_fold_idx, val_fold_idx) in enumerate(
        cv.split(X_all, y_all, groups=groups_all),
        start=1,
    ):
        tr_part = cat_df.iloc[tr_fold_idx]
        val_part = cat_df.iloc[val_fold_idx]

        X_tr, X_val = build_features(tr_part, val_part)
        y_tr = tr_part['final_category']
        y_val = val_part['final_category']

        model = make_model(best_model_type, model_params)
        model.fit(X_tr, y_tr, verbose=False) if best_model_type == 'CatBoost' else model.fit(X_tr, y_tr)
        pred = model.predict(X_val).flatten() if best_model_type == 'CatBoost' else model.predict(X_val)

        cv_results.append({
            'seed': cv_seed,
            'fold': fold,
            'accuracy': accuracy_score(y_val, pred),
            'macro_f1': f1_score(y_val, pred, average='macro'),
        })
        y_true_all.extend(y_val)
        y_pred_all.extend(pred)

cv_results_df = pd.DataFrame(cv_results)
y_true_cv = pd.Series(y_true_all, name='true')
y_pred_cv = pd.Series(y_pred_all, name='pred')

X_full, _, preprocessors = build_features(cat_df, cat_df, return_preprocessors=True)
best_model = make_model(best_model_type, model_params)
best_model.fit(X_full, y_all, verbose=False) if best_model_type == 'CatBoost' else best_model.fit(X_full, y_all)

SVC(C=3.8700335721489627, class_weight='balanced', kernel='linear')

## Результат на тестовой выборке

In [30]:
accuracy_mean = cv_results_df['accuracy'].mean()
accuracy_std = cv_results_df['accuracy'].std()
macro_f1_mean = cv_results_df['macro_f1'].mean()
macro_f1_std = cv_results_df['macro_f1'].std()

print(f'\nBest model type: {best_model_type}')
print(f'CV seeds: {CV_RANDOM_STATES}')
print(f'Accuracy: {accuracy_mean:.4f} +/- {accuracy_std:.4f}')
print(f'Macro F1: {macro_f1_mean:.4f} +/- {macro_f1_std:.4f}')
print('\nFOLD METRICS SUMMARY:\n')
print(cv_results_df[['accuracy', 'macro_f1']].describe())
print('\nCLASSIFICATION REPORT OVER ALL CV PREDICTIONS:\n')
print(classification_report(y_true_cv, y_pred_cv, zero_division=0))
print('\nCONFUSION MATRIX OVER ALL CV PREDICTIONS:\n')
print(confusion_matrix(y_true_cv, y_pred_cv))


Best model type: SVC
CV seeds: [42, 101, 202, 303, 404]
Accuracy: 0.2093 +/- 0.1780
Macro F1: 0.0782 +/- 0.0696

FOLD METRICS SUMMARY:

        accuracy   macro_f1
count  25.000000  25.000000
mean    0.209273   0.078194
std     0.178020   0.069628
min     0.003263   0.000813
25%     0.010060   0.006309
50%     0.214700   0.068403
75%     0.380789   0.121339
max     0.550580   0.213171

CLASSIFICATION REPORT OVER ALL CV PREDICTIONS:

                     precision    recall  f1-score   support

   вопрос по оплате       0.04      0.07      0.05      1850
   жалоба на сервис       0.72      0.26      0.39      1705
    запрос возврата       0.00      0.00      0.00      1940
  запрос документов       0.31      0.64      0.42      1660
       консультация       0.38      0.52      0.44      1945
    ошибка в данных       0.15      0.20      0.17      2050
проблема с доступом       0.00      0.00      0.00      1785
   технический сбой       0.23      0.07      0.10      2065

           

## Сохранение

In [31]:
classification_report_dict = classification_report(
    y_true_cv,
    y_pred_cv,
    zero_division=0,
    output_dict=True,
)
confusion_matrix_values = confusion_matrix(y_true_cv, y_pred_cv)

bundle = {
    'task': 'tickets_category_comment_only',
    'model': best_model,
    'model_type': best_model_type,
    'model_params': model_params,
    'preprocessors': preprocessors,
    'config': {
        'text_column': 'message',
        'target_column': 'final_category',
        'id_column': 'ticket_id',
        'feature_order': ['word_tfidf', 'char_tfidf'],
        'random_state': RANDOM_STATE,
        'n_splits': N_SPLITS,
        'cv_random_states': CV_RANDOM_STATES,
        'bad_stopwords': bad_stopwords,
        'russian_stopwords': russian_stopwords,
        'clean_text_source': CLEAN_TEXT_SOURCE,
        'morph_analyzer': 'pymorphy3.MorphAnalyzer()',
    },
    'metrics': {
        'cv_results': cv_results_df,
        'accuracy_mean': accuracy_mean,
        'accuracy_std': accuracy_std,
        'macro_f1_mean': macro_f1_mean,
        'macro_f1_std': macro_f1_std,
        'classification_report': classification_report_dict,
        'confusion_matrix': confusion_matrix_values.tolist(),
    },
    'metadata': {
        'data_path': 'nerd_analytics_v25.xlsx',
        'tickets_sheet': 'tickets',
        'chat_history_sheet': 'chat_history',
        'n_rows': len(cat_df),
        'classes': sorted(y_all.unique().tolist()),
        'word_features': preprocessors['word_vectorizer'].transform(cat_df['clean_message']).shape[1],
        'char_features': preprocessors['char_vectorizer'].transform(cat_df['clean_message']).shape[1],
    },
}

joblib.dump(bundle, 'tickets_categories_comment_only.pkl')
print('Saved model bundle to: tickets_categories_comment_only.pkl')
print(bundle['metadata'])

Saved model bundle to: tickets_categories_comment_only.pkl
{'data_path': 'nerd_analytics_v25.xlsx', 'tickets_sheet': 'tickets', 'chat_history_sheet': 'chat_history', 'n_rows': 3000, 'classes': ['вопрос по оплате', 'жалоба на сервис', 'запрос возврата', 'запрос документов', 'консультация', 'ошибка в данных', 'проблема с доступом', 'технический сбой'], 'word_features': 148, 'char_features': 1213}
